In [1]:
from datetime import datetime
from collections import OrderedDict
import pandas as pd

import torch
import selfies as sf
import warnings
warnings.filterwarnings('ignore')

from input_output import *

from network.selfies2selfies_netwrok import *
from data_utils import *

%load_ext autoreload
%autoreload 2

In [2]:
path_init = '/home/rkmvu/Dataset/selfies/zinc/'
# GPU device parameter
gpu_device_id = 0# GPU number for multiple GPUs (pytorch takes default 0 or which is availabe next)
device = torch.device("cuda:"+str(gpu_device_id) if torch.cuda.is_available() else "cpu")

# dataset parameters
train_ratio = 0.5
val_ratio = 0.2
test_ratio = 1 - train_ratio - val_ratio
num_tokens = 58
selfies_max_len = 128
flag_selfies_tokens = True
flag_extra_data = True
flag_extra_tokens = False
num_extra_tokens = 105
req_selfies_max_len = 128#150
num_train_samples = 12000000#18000000#12941195#10000
num_train_test_samples = 100000#1000#10000#

In [3]:
#read selfies tokens
input_filename = ''.join([
    path_init,
    'tokens_selfies_',
    'train_', str(train_ratio).replace('.', '_'),
    '_val_', str(val_ratio).replace('.', '_'),
    '_test_', str(test_ratio).replace('.', '_'), 
    '_', str(num_tokens),
    '.csv'
])
tokens_selfies = _read_tokens_selfies(input_filename)
if flag_selfies_tokens:
    tokens_selfies = list(set(tokens_selfies).union(sf.get_semantic_robust_alphabet()))
dict_selfies_tokens = _get_dict_selfies(tokens_selfies)

#read selfies max length
input_filename = ''.join([
    path_init,
    'max_len_selfies_',
    'train_', str(train_ratio).replace('.', '_'),
    '_val_', str(val_ratio).replace('.', '_'),
    '_test_', str(test_ratio).replace('.', '_'), 
    '_', str(selfies_max_len),
    '.txt'
])
selfies_max_len = _read_max_len_selfies(input_filename)

Reading data from "/home/rkmvu/Dataset/selfies/zinc/tokens_selfies_train_0_5_val_0_2_test_0_3_58.csv"
----------------------------------------------------------------------
Done!


In [4]:
len(tokens_selfies)

87

In [5]:
## VAE_GRU parameters
dims_input_data = (selfies_max_len, len(tokens_selfies)) 
dims_output_data = (selfies_max_len, len(tokens_selfies))
num_kernels = [9, 9, 10]#[25, 21, 17]#[11, 13, 15]#[15, 17, 19]#[11, 13, 15]#
size_kernels = [9, 9, 11]#[13, 11, 9]#[9, 9, 11]#[15, 13, 11]#[9, 9, 11]#
num_fc_layer_encoder = 0
dropout_prob = 0.0
dims_latent = 300
act_func = 'relu'
scale_latent_space = 1e-2
num_fc_layer_decoder = 0
gru_hidden_size = 500
num_gru = 4
layer_type = '1dcnn_gru'
weight_init = 'xvr_unifrm'
loss_type = 'bce_kld'#'bce_kld_uniform'#
solver_type = 'adam'
num_epoch = 500
batch_size = 128
learning_rate = 1e-4
save_result_ateach_epoch = 50
result_savepath = ''.join([
    'cache/selfie2selfies_vae/', 
    'train_', str(train_ratio).replace('.', '_'),
    '_val_', str(val_ratio).replace('.', '_'),
    '_test_', str(test_ratio).replace('.', '_'), 
    '_nt_', str(num_tokens),
    '_ml_', str(selfies_max_len),
    '_slft_', str(flag_selfies_tokens),
    '_exd_'+str(flag_extra_data)+'_ext_'+str(flag_extra_tokens)+'_nt_'+str(num_extra_tokens)+'_rsl_'+str(req_selfies_max_len) if flag_extra_data else '_exd_'+str(flag_extra_data),
    '_nts_', str(num_train_samples),
    '_ntts_', str(num_train_test_samples),
    '/'
]).replace('.','_')
#+++++++++++++++++++++++++++++++++++++++++++++++++++


In [6]:

selfies_vae_model = net_selfies2selfies(
    dims_input_data, 
    dims_output_data, 
    max_string_len=selfies_max_len,
    num_kernels=num_kernels, 
    size_kernels=size_kernels, 
    num_fc_layer_encoder=num_fc_layer_encoder, 
    dropout_prob=dropout_prob, 
    dims_latent=dims_latent, 
    act_func=act_func, 
    scale_latent_space=scale_latent_space, 
    num_fc_layer_decoder=num_fc_layer_decoder, 
    gru_hidden_size=gru_hidden_size, 
    num_gru=num_gru, 
    layer_type=layer_type, 
    device=device, 
    weight_init=weight_init, 
    loss_type=loss_type, 
    solver_type=solver_type, 
    num_epoch=num_epoch, 
    batch_size=batch_size, 
    learning_rate=learning_rate, 
    save_result_ateach_epoch=save_result_ateach_epoch, 
    result_savepath=result_savepath
)

selfies_vae_model.load_test_network()# load pre-trained best network

----------------------------------------------------------------------
Network summary
Layer (type:depth-idx)                   Output Shape              Param #
CNN1D_GRU_VAE                            [128, 128, 87]            --
├─Sequential: 1-1                        [128, 10, 61]             --
│    └─Conv1d: 2-1                       [128, 9, 79]              10,377
│    └─ReLU: 2-2                         [128, 9, 79]              --
│    └─Conv1d: 2-3                       [128, 9, 71]              738
│    └─ReLU: 2-4                         [128, 9, 71]              --
│    └─Conv1d: 2-5                       [128, 10, 61]             1,000
│    └─ReLU: 2-6                         [128, 10, 61]             --
├─Sequential: 1-2                        [128, 300]                --
│    └─Linear: 2-7                       [128, 300]                183,300
├─Sequential: 1-3                        [128, 300]                --
│    └─Linear: 2-8                       [128, 300]    

## **Test the model for representation**

In [7]:
def encode(smiles_or_selfies):
    z =  selfies_vae_model.selfies2latent_vector(selfies=smiles_or_selfies, 
                                                   dict_selfies_tokens=dict_selfies_tokens)
    return z

def decode(x_latent):
    x = selfies_vae_model.latent_vector2selfies(x_latent=x_latent, 
                                                dict_selfies_tokens=dict_selfies_tokens)
    return x

def name2selfies(name):
    row = df_temp[df_temp['name'] == name].iloc[0]
    return row['selfies']

def selfies2name(selfies):
    row = df_temp[df_temp['selfies'] == selfies].iloc[0]
    return row['name']

def smi2nneigh(selfies, indices, n_neigh=10):

    idx = selfies_main.index(selfies)
    neigh_idx = indices[idx][1:n_neigh+1]
    nn_selfies = [selfies_main[i] for i in neigh_idx]
    nn_names = [selfies2name(x) for x in nn_selfies]
    
    return {'selfies':nn_selfies, 'name':nn_names}


In [8]:
import selfies as sf

smiles2selfie_fn = lambda x: sf.encoder(x)
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)

df = pd.read_csv('/home/rkmvu/Dataset/selfies/zinc/properties_dtratio_0.001_test_0_3_canon_smiles_selfies_with_descriptors_train_0_5_val_0_2_test_0_3.csv')
mask = df['SMILES'].apply(smiles_clean_fn)
df = df[mask].copy()
df['selfies'] = df['SMILES'].apply(smiles2selfie_fn)
selfies_main = df['selfies'].tolist()
df.head(2)

,SMILES,MolWt,TPSA,EState_VSA1,NHOHCount,MolLogP,fr_COO,nAcid,ATSC1c,ATSC1se,...,fr_halogen,fr_ketone,fr_nitro,fr_nitro_arom,fr_nitroso,fr_phenol,fr_sulfone,AUTOCORR2D_156,nBase,selfies
0,COc1ccc(CN2CCC3CN(C(=O)C(C)C(C)(C)C)C3C2)nn1,346.475,58.56,0.000000,0,2.2001,0,0,-0.330215,0.267548,...,0,0,0,0,0,0,0,1.405,1,[C][O][C][=C][C][=C][Branch2][Ring1][S][C][N][...
1,C#CCOC(C)C(=O)N1CC2(CCCN2C(=O)c2ccc(CO)o2)C1,346.383,83.22,6.103966,1,0.6272,0,0,-0.612666,-0.332817,...,0,0,0,0,0,0,0,1.405,0,[C][#C][C][O][C][Branch1][C][C][C][=Branch1][C...


In [9]:
from data_utils import _check_validity_selfies

z = encode(smiles_or_selfies=selfies_main)
selfies_recon = decode(x_latent=z)

exact = [x==y for x, y in zip(selfies_main, selfies_recon)]
selfiles, check_ids, smiles_decode = _check_validity_selfies(selfiles=selfies_recon)

print('='*80)
print(f'Number of SMILES: {len(selfies_main)}')
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(check_ids)/z.shape[0]}')
print('='*80)


<===doing selfies reconstruction===>
----------------------------------------------------------------------


 19%|█▊        | 10/54 [00:00<00:00, 94.05it/s]

100%|██████████| 54/54 [00:00<00:00, 83.36it/s] 


<===doing selfies reconstruction===>
----------------------------------------------------------------------


100%|██████████| 54/54 [00:01<00:00, 48.54it/s]


<===checking selfies validity===>
----------------------------------------------------------------------


100%|██████████| 6813/6813 [00:06<00:00, 1028.40it/s]

Number of SMILES: 6813
Exact reconstruction: 0.9856157346249816
Valid reconstruction: 1.0


## **Test the model for representation**

In [10]:
import selfies as sf
from rdkit import Chem

smiles2selfie_fn = lambda x: sf.encoder(x)
selfie2smiles_fn = lambda x: sf.decoder(x)
canonical_fn = lambda smi: Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=False)
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)
len_filter_fn = lambda x: sf.len_selfies(x)<=selfies_max_len

#### **Preprocess drugs**

In [11]:
from data_utils import _check_smiles_validity

df = pd.read_csv('/home/rkmvu/Dataset/selfies/drug/mdrug.csv')
_, valid_mask = _check_smiles_validity(smiles=df['smiles'].tolist())
df = df[valid_mask].copy()
df['smiles'] = df['smiles'].apply(canonical_fn)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

len_mask = df['selfies'].apply(len_filter_fn)
df = df[len_mask]

df_temp = df.copy()
mask = df_temp['smiles'].apply(smiles_clean_fn)
df_temp = df_temp[mask]
df_temp = df_temp.drop_duplicates(subset=['smiles'])
selfies_main = df_temp['selfies'].tolist()
print(f'Number of selfies: {len(selfies_main)}')
df_temp

<===checking smiles validity===>


 34%|███▍      | 473/1381 [00:00<00:00, 4727.05it/s]

100%|██████████| 1381/1381 [00:00<00:00, 4740.47it/s]


Number of selfies: 1089


,name,smiles,InChl,type,selfies
0,Abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,InChI=1S/C14H18N6O/c15-14-18-12(17-9-2-3-9)11-...,Drug,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...
1,Abiraterone,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,InChI=1S/C26H33NO2/c1-17(28)29-20-10-12-25(2)1...,Drug,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...
2,Acamprosate,CC(=O)NCCCS(=O)(=O)O,"InChI=1S/C5H11NO4S/c1-5(7)6-3-2-4-11(8,9)10/h2...",Drug,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...
3,Acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,InChI=1S/C25H43NO18/c1-6-11(26-8-2-7(3-27)12(3...,Drug,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...
4,Acebutolol,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,InChI=1S/C18H28N2O4/c1-5-6-18(23)20-14-7-8-17(...,Drug,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...
...,...,...,...,...,...
1374,Ziprasidone,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,InChI=1S/C21H21ClN4OS/c22-17-13-18-15(12-20(27...,Drug,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...
1375,Zoledronate,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,"InChI=1S/C5H10N2O7P2/c8-5(15(9,10)11,16(12,13)...",Drug,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...
1377,Zolpidem,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,InChI=1S/C19H21N3O/c1-13-5-8-15(9-6-13)19-16(1...,Drug,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...
1378,Zonisamide,NS(=O)(=O)Cc1noc2ccccc12,"InChI=1S/C8H8N2O3S/c9-14(11,12)5-7-6-3-1-2-4-8...",Drug,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...


### **Finding Nearest Neighbours**

In [12]:
from sklearn.neighbors import NearestNeighbors

z = encode(smiles_or_selfies=selfies_main)
n_neigh = NearestNeighbors(n_neighbors=60, metric='euclidean')
n_neigh.fit(z)

<===doing selfies reconstruction===>
----------------------------------------------------------------------


100%|██████████| 9/9 [00:00<00:00, 111.28it/s]


NearestNeighbors(metric='euclidean', n_neighbors=60)

In [13]:
distances, indices = n_neigh.kneighbors(z)
distances
indices

array([[   0,  246,  718, ...,  310,  783,  652],
       [   1,  199,  315, ...,  896,  681, 1028],
       [   2,   12,  652, ...,  805,  543,  832],
       ...,
       [1086, 1032,  309, ..., 1027,  806,  120],
       [1087,  788,  792, ...,  611, 1050,  120],
       [1088,  787,  441, ...,  638,  867,  869]])

In [16]:
df = pd.DataFrame(z, columns=[f'dim_{i}' for i in range(z.shape[1])])
df.insert(0, 'smiles', 'None')
df.insert(0, 'selfies', selfies_main)
df.insert(0, 'name', 'None')
df['name'] = df['selfies'].apply(selfies2name)
df['smiles'] = df['selfies'].apply(selfie2smiles_fn).apply(canonical_fn)

df2 = pd.read_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_properties.csv')
df3 = pd.merge(df, df2, on='smiles')
# df3.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_props_with_embeds_evslfi.csv', index=False)
df3#.head(2)

,name,selfies,smiles,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,...,VSA_EState3,NHOHCount,NumHDonors,NumHAcceptor,NumRotatableBonds,MolLogP,ATSC1pe,ATSC1are,AATSC1dv,AATSC1are
0,Abacavir,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,0.010825,0.045238,0.007190,0.012951,-0.000592,0.029317,-0.026538,...,12.615719,4,3,7,4,1.09230,-0.478300,-0.657070,1.039823,-0.015645
1,Abiraterone,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,0.027688,0.006653,0.003370,0.007166,0.038433,-0.038168,-0.007706,...,0.000000,0,0,3,2,5.96940,0.296168,0.241524,1.157286,0.003659
2,Acamprosate,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...,CC(=O)NCCCS(=O)(=O)O,-0.002691,-0.013326,-0.051155,0.004678,0.017658,0.006438,0.046674,...,2.403611,2,2,3,4,-0.59960,-0.423500,-0.761825,-0.369321,-0.036277
3,Acarbose,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,-0.129665,0.080479,0.089610,-0.048958,0.053028,0.127103,0.073762,...,135.592960,14,14,19,9,-8.56450,-4.505442,-5.305658,-0.192650,-0.058952
4,Acebutolol,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,0.020988,0.010750,0.004093,-0.000291,-0.015668,0.003815,-0.045177,...,15.766019,3,3,5,10,2.36550,-0.276185,-0.373677,1.298077,-0.007186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1084,Ziprasidone,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,-0.015821,0.042206,0.044328,-0.004296,0.000999,0.018327,0.054155,...,4.859700,1,1,5,4,3.80900,0.140586,0.027960,0.820617,0.000528
1085,Zoledronate,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,-0.001022,0.007649,-0.037032,-0.000105,-0.028872,0.029461,-0.000752,...,6.037515,5,5,5,4,-1.11540,-3.912300,-4.806854,-2.436445,-0.184879
1086,Zolpidem,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,0.018451,-0.010696,-0.022666,0.002127,-0.012304,0.020004,-0.004070,...,0.000000,0,0,3,3,3.24884,0.303576,0.245848,1.704815,0.005345
1087,Zonisamide,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...,NS(=O)(=O)Cc1noc2ccccc12,0.032213,-0.052196,-0.015258,0.001232,0.031746,0.035336,0.020020,...,9.236725,2,1,4,2,0.61630,0.064800,-0.113569,0.317683,-0.004938


In [15]:
name2selfies('Abacavir')

'[N][C][=N][C][Branch1][#Branch1][N][C][C][C][Ring1][Ring1][=C][N][=C][N][Branch1][N][C][C][=C][C][Branch1][Ring1][C][O][C][Ring1][#Branch1][C][Ring1][N][=N][Ring2][Ring1][Ring2]'

In [16]:
selfies2name(name2selfies('Abacavir'))

'Abacavir'

In [17]:
smi2nneigh(name2selfies('Loxapine'),indices=indices)

{'selfies': ['[C][N][C][C][C][Branch2][Ring1][=Branch1][C][N][C][=C][C][=C][C][=C][Ring1][=Branch1][S][C][=C][C][=C][C][=C][Ring1][=Branch1][Ring1][=C][C][Ring2][Ring1][Ring2]',
  '[C][N][C][C][N][Branch2][Ring1][=Branch2][C][=N][C][=C][C][Branch1][C][Cl][=C][C][=C][Ring1][#Branch1][N][C][=C][C][=C][C][=C][Ring1][=Branch1][Ring1][S][C][C][Ring2][Ring1][=Branch1]',
  '[C][O][C][=C][C][=C][Branch2][Ring1][Branch2][C][C][=N][C][=C][C][=C][C][Branch1][Ring1][O][C][=C][Branch1][Ring1][O][C][C][=C][Ring1][=C][Ring1][#Branch2][C][=C][Ring2][Ring1][Branch1][O][C]',
  '[C][N][C][C][C][=Branch2][Ring1][Branch2][=C][C][=C][C][=C][C][=C][Ring1][=Branch1][C][C][=Branch1][C][=O][C][S][C][=C][C][=Ring1][Branch1][Ring1][#C][C][C][Ring2][Ring1][Branch1]',
  '[C][N][C][C][C][=Branch2][Ring1][=Branch2][=C][C][=C][C][=C][C][=C][Ring1][=Branch1][C][C][N][C][Branch1][Ring1][C][=O][=C][N][=C][Ring1][#Branch1][Ring1][S][C][C][Ring2][Ring1][=Branch1]',
  '[C][N][C][=Branch1][C][=O][C][N][=C][Branch1][#Branch2]

## **Nearest Neighbour Analysis**

In [17]:
import numpy as np

top_drugs = ["Metformin", "Amoxicillin", "Atorvastatin", "Amlodipine", "Acetaminophen", "Imatinib", "Clozapine", 
             "Ibuprofen", "Azithromycin", "Doxycycline"]
top_drugs = df_temp['name']
n_neigh=50

all_out = {'drug_evslfi':[], 'nn_idx_evslfi':[], 'smiles_evslfi':[], 'names_evslfi':[]}
for drug in top_drugs:
    _smiles = name2selfies(drug)
    out = smi2nneigh(_smiles, indices=indices, n_neigh=n_neigh)
    drug_name = [drug]*n_neigh
    idx = np.arange(1, n_neigh+1)
    out = {'drug':drug_name, 'nn_idx':idx, **out}
    all_out['drug_evslfi'].extend(out['drug'])
    all_out['nn_idx_evslfi'].extend(out['nn_idx'])
    all_out['smiles_evslfi'].extend(map(selfie2smiles_fn, out['selfies']))
    all_out['names_evslfi'].extend(out['name'])

In [20]:
all_out_df = pd.DataFrame(all_out)
# all_out_df.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/near_smiles_evslfi.csv', index=False)
all_out_df

,drug_evslfi,nn_idx_evslfi,smiles_evslfi,names_evslfi
0,Abacavir,1,NC1=NC(Cl)=NC2=C1N=CN2C3OC(CO)C(O)C3F,Clofarabine
1,Abacavir,2,COC1=NC(N)=NC2=C1N=CN2C3OC(CO)C(O)C3O,Nelarabine
2,Abacavir,3,NC1=NC(Cl)=NC2=C1N=CN2C3CC(O)C(CO)O3,Cladribine
3,Abacavir,4,OCC1OC(N2C=NC3=C2N=CNCC3O)CC1O,Pentostatin
4,Abacavir,5,CCC1C(=O)OCC1CC2=CN=CN2C,Pilocarpine
...,...,...,...,...
54445,Zuclopenthixol,46,COC1=CC=C2C3=C1OC4CC(O)C=CC34CCN(C)C2,Galantamine
54446,Zuclopenthixol,47,CN(C)CCC=C1C2=CC=CC=C2C=CC3=CC=CC=C31,Cyclobenzaprine
54447,Zuclopenthixol,48,CC(C)NCC(O)C1=CC(O)=CC(O)=C1,Orciprenaline
54448,Zuclopenthixol,49,CN(C)CCOC(=O)C(C1=CC=CC=C1)C2(O)CCCC2,Cyclopentolate


### **Reconstruction accuracy**

In [20]:
z = encode(smiles_or_selfies=selfies_main)
selfies_recon = decode(x_latent=z)
print(z.shape)

exact = [x==y for x, y in zip(selfies_main, selfies_recon)]
smis, checks,_ = _check_validity_selfies(selfiles=selfies_recon)

print('='*80)
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(checks)/z.shape[0]}')
print('='*80)


<===doing selfies reconstruction===>
----------------------------------------------------------------------


100%|██████████| 9/9 [00:00<00:00, 121.30it/s]


<===doing selfies reconstruction===>
----------------------------------------------------------------------


 67%|██████▋   | 6/9 [00:00<00:00, 50.88it/s]

100%|██████████| 9/9 [00:00<00:00, 44.98it/s]


(1089, 100)
<===checking selfies validity===>
----------------------------------------------------------------------


100%|██████████| 1089/1089 [00:00<00:00, 1116.50it/s]

Exact reconstruction: 0.4986225895316804
Valid reconstruction: 1.0
